In [2]:
# 03_audit_aml — Cell 1: load HI-Small into DuckDB with explicit types (bank/account kept as text)
import sys; sys.path.insert(0, "..")
import pandas as pd
import numpy as np
from src.config import PATHS, SEED, Timer, duckdb_connect

AML_CSV = (PATHS["raw"] / "amlworld" / "HI-Small_Trans.csv").as_posix()
N_EXPECTED = 5_078_345                      # row count verified in 01_download

# Names replace the CSV header by position (same order as printed in 01_download)
COLS = {"ts_raw": "VARCHAR", "from_bank": "VARCHAR", "from_acct": "VARCHAR",
        "to_bank": "VARCHAR", "to_acct": "VARCHAR", "amt_received": "DOUBLE",
        "recv_ccy": "VARCHAR", "amt_paid": "DOUBLE", "pay_ccy": "VARCHAR",
        "pay_format": "VARCHAR", "is_laundering": "TINYINT"}

con = duckdb_connect()
with Timer("CSV -> DuckDB table aml"):
    con.execute(f"""
        CREATE OR REPLACE TABLE aml AS
        SELECT strptime(ts_raw, '%Y/%m/%d %H:%M') AS ts,
               * EXCLUDE (ts_raw),
               from_bank || '_' || from_acct AS src,     -- unique account key = bank + account
               to_bank   || '_' || to_acct   AS dst
        FROM read_csv('{AML_CSV}', header = true, columns = {COLS})
    """)

n, n_null, t0, t1, n_pos = con.execute("""
    SELECT count(*), count(*) FILTER (WHERE src IS NULL OR dst IS NULL),
           min(ts), max(ts), sum(is_laundering) FROM aml""").fetchone()
assert n == N_EXPECTED, n
print(f"rows = {n:,} | NULL src/dst = {n_null} | laundering = {n_pos:,} ({100*n_pos/n:.4f}%)")
print(f"time range: {t0}  ->  {t1}   ({(t1 - t0).total_seconds()/86400:.2f} days)\n")
print(con.execute("DESCRIBE aml").df()[["column_name", "column_type"]].to_string(index=False))
print()
print(con.execute("SELECT * FROM aml ORDER BY ts LIMIT 3").df().to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[CSV -> DuckDB table aml] wall = 12.1 s | RSS now = 0.40 GB | peak RSS = 1.17 GB
rows = 5,078,345 | NULL src/dst = 0 | laundering = 5,177 (0.1019%)
time range: 2022-09-01 00:00:00  ->  2022-09-18 16:18:00   (17.68 days)

  column_name column_type
           ts   TIMESTAMP
    from_bank     VARCHAR
    from_acct     VARCHAR
      to_bank     VARCHAR
      to_acct     VARCHAR
 amt_received      DOUBLE
     recv_ccy     VARCHAR
     amt_paid      DOUBLE
      pay_ccy     VARCHAR
   pay_format     VARCHAR
is_laundering     TINYINT
          src     VARCHAR
          dst     VARCHAR

          ts from_bank  from_acct to_bank    to_acct  amt_received   recv_ccy  amt_paid    pay_ccy    pay_format  is_laundering              src              dst
0 2022-09-01     03209  8000F4670   03209  8000F4670      14675.57  US Dollar  14675.57  US Dollar  Reinvestment              0  03209_8000F4670  03209_8000F4670
1 2022-09-01     01420  8005DFEB0   01420  8005DFEB0        897.37  US Dollar    897.37  U

In [3]:
# 03_audit_aml — Cell 2: activity and laundering per day; 60/20/20 chronological cut points
daily = con.execute("""
    SELECT CAST(ts AS DATE) AS day, count(*) AS n_tx, sum(is_laundering) AS n_laund,
           round(100.0 * avg(is_laundering), 4) AS laund_pct
    FROM aml GROUP BY 1 ORDER BY 1
""").df()
daily["cum_share_pct"] = (100 * daily["n_tx"].cumsum() / daily["n_tx"].sum()).round(2)
print(daily.to_string(index=False))

# Cut points at 60% and 80% of transactions, ordered by time
cut60, cut80 = con.execute("SELECT quantile_disc(ts, [0.6, 0.8]) FROM aml").fetchone()[0]
print(f"\n60% cut: {cut60} | 80% cut: {cut80}\n")

split = con.execute(f"""
    SELECT CASE WHEN ts <  TIMESTAMP '{cut60}' THEN 'train'
                WHEN ts <  TIMESTAMP '{cut80}' THEN 'valid'
                ELSE 'test' END AS split,
           count(*) AS n, sum(is_laundering) AS n_laund,
           round(100.0 * avg(is_laundering), 4) AS laund_pct,
           min(ts) AS t_min, max(ts) AS t_max,
           round(date_diff('minute', min(ts), max(ts)) / 1440.0, 2) AS span_days
    FROM aml GROUP BY 1 ORDER BY t_min
""").df()
print(split.to_string(index=False))

       day    n_tx  n_laund  laund_pct  cum_share_pct
2022-09-01 1114921    322.0     0.0289          21.95
2022-09-02  754449    408.0     0.0541          36.81
2022-09-03  207382    391.0     0.1885          40.89
2022-09-04  207430    407.0     0.1962          44.98
2022-09-05  482650    471.0     0.0976          54.48
2022-09-06  482089    531.0     0.1101          63.98
2022-09-07  482751    497.0     0.1030          73.48
2022-09-08  482773    539.0     0.1116          82.99
2022-09-09  654467    514.0     0.0785          95.88
2022-09-10  208325    442.0     0.2122          99.98
2022-09-11     396    232.0    58.5859          99.99
2022-09-12     281    170.0    60.4982          99.99
2022-09-13     184    106.0    57.6087         100.00
2022-09-14     121     70.0    57.8512         100.00
2022-09-15      46     28.0    60.8696         100.00
2022-09-16      46     26.0    56.5217         100.00
2022-09-17      23     15.0    65.2174         100.00
2022-09-18      11      8.0 

In [4]:
# 03_audit_aml — Cell 3: tail after 2022-09-10 and test_core statistics (evidence for DECISION #6)
TAIL_START = "2022-09-11"
print("cut80 from Cell 2:", cut80)

print("\n=== test parts: test_core (cut80 <= ts < 2022-09-11) vs tail ===")
print(con.execute(f"""
    SELECT CASE WHEN ts < TIMESTAMP '{TAIL_START}' THEN 'test_core' ELSE 'tail' END AS part,
           count(*) AS n, sum(is_laundering) AS n_laund,
           round(100.0 * avg(is_laundering), 4) AS laund_pct,
           count(DISTINCT date_trunc('hour', ts)) AS hour_blocks,
           min(ts) AS first_ts, max(ts) AS last_ts
    FROM aml WHERE ts >= TIMESTAMP '{cut80}' GROUP BY 1 ORDER BY 1 DESC
""").df().to_string(index=False))

print("\n=== tail: payment formats ===")
print(con.execute(f"""
    SELECT pay_format, count(*) AS n, sum(is_laundering) AS n_laund
    FROM aml WHERE ts >= TIMESTAMP '{TAIL_START}' GROUP BY 1 ORDER BY n DESC
""").df().to_string(index=False))

print("\n=== tail: by hour of day ===")
print(con.execute(f"""
    SELECT hour(ts) AS h, count(*) AS n, sum(is_laundering) AS n_laund
    FROM aml WHERE ts >= TIMESTAMP '{TAIL_START}' GROUP BY 1 ORDER BY 1
""").df().T.to_string(header=False))

print("\n=== tail source accounts: seen before the tail? ===")
print(con.execute(f"""
    WITH tail_src AS (SELECT DISTINCT src FROM aml WHERE ts >= TIMESTAMP '{TAIL_START}'),
         pre_acc  AS (SELECT src AS acc FROM aml WHERE ts < TIMESTAMP '{TAIL_START}'
                      UNION SELECT dst FROM aml WHERE ts < TIMESTAMP '{TAIL_START}')
    SELECT count(*) AS tail_src_accounts,
           count(*) FILTER (WHERE src IN (SELECT acc FROM pre_acc)) AS seen_before_tail
    FROM tail_src
""").df().to_string(index=False))

cut80 from Cell 2: 2022-09-08 16:12:00

=== test parts: test_core (cut80 <= ts < 2022-09-11) vs tail ===
     part       n  n_laund  laund_pct  hour_blocks            first_ts             last_ts
test_core 1014774   1143.0     0.1126           56 2022-09-08 16:12:00 2022-09-10 23:59:00
     tail    1108    655.0    59.1155          145 2022-09-11 00:05:00 2022-09-18 16:18:00

=== tail: payment formats ===
pay_format    n  n_laund
       ACH 1108    655.0

=== tail: by hour of day ===
h         0.0   1.0   2.0   3.0   4.0   5.0   6.0   7.0   8.0   9.0  10.0  11.0  12.0  13.0  14.0  15.0  16.0  17.0  18.0  19.0  20.0  21.0  22.0  23.0
n        22.0  21.0  29.0  32.0  29.0  51.0  45.0  34.0  61.0  52.0  61.0  79.0  75.0  87.0  80.0  72.0  58.0  46.0  69.0  50.0  14.0  13.0  13.0  15.0
n_laund  14.0  14.0  16.0  19.0  17.0  30.0  26.0  21.0  36.0  33.0  35.0  46.0  48.0  49.0  49.0  41.0  35.0  28.0  39.0  28.0   9.0   8.0   6.0   8.0

=== tail source accounts: seen before the tail? ===
 t